In [1]:
from pathlib import Path
import sys
import numpy as np

folder = Path.cwd()
candidates = [
    folder / "src",
    folder.parent / "src",
    folder / "sib" / "src"
]

src = next(path for path in candidates if (path / "si").is_dir())
sys.path.insert(0, str(src))

from si.neural_networks.layers import DenseLayer

In [2]:
layer = DenseLayer(4, (3,)).initialize()

out = layer.forward_propagation(np.random.rand(10, 3))

assert out.shape == (10, 4)
assert layer.parameters() == 3 * 4 + 4
assert layer.output_shape() == (4,)

print("Todos os testes passaram.")

Todos os testes passaram.


In [3]:
class NoUpdate:
    def update(self, w, grad):
        self.gradient = grad.copy()
        return w


mse = lambda Y, P: np.mean((Y - P) ** 2)
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size

np.random.seed(0)

X = np.random.rand(4, 3)
Y = np.random.rand(4, 2)

layer = DenseLayer(2, (3,)).initialize(NoUpdate())

# Forward e backward do método que implementaste
out = layer.forward_propagation(X, training=True)
output_error = mse_derivative(Y, out)
input_error = layer.backward_propagation(output_error)

analytic = layer.weights_optimizer.gradient.copy()

# Aproximar o gradiente alterando ligeiramente cada peso
numeric = np.zeros_like(layer.weights)
eps = 1e-6

for i in range(3):
    for j in range(2):
        original = layer.weights[i, j]

        layer.weights[i, j] = original + eps
        lp = mse(Y, layer.forward_propagation(X, training=True))

        layer.weights[i, j] = original - eps
        lm = mse(Y, layer.forward_propagation(X, training=True))

        layer.weights[i, j] = original
        numeric[i, j] = (lp - lm) / (2 * eps)

difference = np.abs(numeric - analytic).max()
print("Diferença máxima:", difference)

np.testing.assert_allclose(
    analytic, numeric, rtol=1e-5, atol=1e-7
)

print("Teste dos gradientes dos pesos passou.")

Diferença máxima: 2.154523781605633e-11
Teste dos gradientes dos pesos passou.


In [4]:
import numpy as np

from si.neural_networks.activation import (
    ReLUActivation,
    SigmoidActivation
)

x = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7])
eps = 1e-6

for activation in [ReLUActivation(), SigmoidActivation()]:

    numeric = (
        activation.activation_function(x + eps)
        - activation.activation_function(x - eps)
    ) / (2 * eps)

    analytic = activation.derivative(x)

    difference = np.abs(numeric - analytic).max()

    print(activation.layer_name(), "— diferença máxima:", difference)

    np.testing.assert_allclose(
        analytic, numeric, rtol=1e-5, atol=1e-7
    )

print("Testes das derivadas passaram.")

ReLUActivation — diferença máxima: 1.397779669787269e-10
SigmoidActivation — diferença máxima: 6.56039597646263e-11
Testes das derivadas passaram.


In [5]:
import numpy as np

from si.neural_networks.layers import Dropout

np.random.seed(0)

X = np.random.rand(3, 4)

dropout = Dropout(probability=0.5, input_shape=(4,))

# Treino
output_train = dropout.forward_propagation(X, training=True)

print("Entrada:")
print(X)

print("\nMáscara:")
print(dropout.mask)

print("\nSaída durante o treino:")
print(output_train)

# Backward associado ao forward de treino
output_error = np.ones_like(X)
input_error = dropout.backward_propagation(output_error)

print("\nGradiente devolvido pelo backward:")
print(input_error)

# Avaliação
output_eval = dropout.forward_propagation(X, training=False)

print("\nSaída durante a avaliação:")
print(output_eval)

np.testing.assert_array_equal(output_eval, X)

print("\nTeste de avaliação passou.")

Entrada:
[[0.5488135  0.71518937 0.60276338 0.54488318]
 [0.4236548  0.64589411 0.43758721 0.891773  ]
 [0.96366276 0.38344152 0.79172504 0.52889492]]

Máscara:
[[1 1 0 0]
 [0 1 1 1]
 [1 1 0 1]]

Saída durante o treino:
[[1.09762701 1.43037873 0.         0.        ]
 [0.         1.29178823 0.87517442 1.783546  ]
 [1.92732552 0.76688304 0.         1.05778984]]

Gradiente devolvido pelo backward:
[[2. 2. 0. 0.]
 [0. 2. 2. 2.]
 [2. 2. 0. 2.]]

Saída durante a avaliação:
[[0.5488135  0.71518937 0.60276338 0.54488318]
 [0.4236548  0.64589411 0.43758721 0.891773  ]
 [0.96366276 0.38344152 0.79172504 0.52889492]]

Teste de avaliação passou.


In [6]:
import numpy as np

from si.neural_networks.activation import (
    TanhActivation,
    SoftmaxActivation
)

X = np.array([
    [2.0, 1.0, 0.1],
    [1.0, 2.0, 3.0]
])

output_error = np.array([
    [0.3, -0.4, 0.2],
    [-0.2, 0.7, 0.5]
])

eps = 1e-6

for activation in [TanhActivation(), SoftmaxActivation()]:

    output = activation.forward_propagation(X)
    analytic = activation.backward_propagation(output_error)

    numeric = np.zeros_like(X)

    # Alterar uma entrada de cada vez
    for index in np.ndindex(X.shape):
        X_plus = X.copy()
        X_minus = X.copy()

        X_plus[index] += eps
        X_minus[index] -= eps

        lp = np.sum(
            activation.activation_function(X_plus) * output_error
        )

        lm = np.sum(
            activation.activation_function(X_minus) * output_error
        )

        numeric[index] = (lp - lm) / (2 * eps)

    difference = np.abs(analytic - numeric).max()

    print(activation.layer_name(), "— diferença máxima:", difference)

    np.testing.assert_allclose(
        analytic, numeric, rtol=1e-5, atol=1e-7
    )

    if isinstance(activation, SoftmaxActivation):
        print("Probabilidades:")
        print(output)

        print("Soma de cada linha:", output.sum(axis=-1))

        np.testing.assert_allclose(
            output.sum(axis=-1), np.ones(X.shape[0])
        )

print("Testes de Tanh e Softmax passaram.")

TanhActivation — diferença máxima: 1.011104880377367e-10
SoftmaxActivation — diferença máxima: 6.21212248308467e-11
Probabilidades:
[[0.65900114 0.24243297 0.09856589]
 [0.09003057 0.24472847 0.66524096]]
Soma de cada linha: [1. 1.]
Testes de Tanh e Softmax passaram.


In [7]:
import numpy as np

from si.neural_networks.losses import MeanSquaredError

y_true = np.array([1.0, 2.0, 3.0])
y_pred = np.array([1.0, 2.5, 2.0])

mse = MeanSquaredError()

error = mse.loss(y_true, y_pred)
gradient = mse.derivative(y_true, y_pred)

print("Erro MSE:", error)
print("Derivada:", gradient)

np.testing.assert_allclose(error, 5 / 12)

np.testing.assert_allclose(
    gradient,
    np.array([0.0, 1 / 3, -2 / 3])
)

print("Teste da MeanSquaredError passou.")

Erro MSE: 0.4166666666666667
Derivada: [ 0.          0.33333333 -0.66666667]
Teste da MeanSquaredError passou.


In [8]:
import numpy as np

from si.neural_networks.losses import BinaryCrossEntropy

y_true = np.array([0.0, 1.0])
y_pred = np.array([0.25, 0.75])

bce = BinaryCrossEntropy()

error = bce.loss(y_true, y_pred)
gradient = bce.derivative(y_true, y_pred)

print("Erro BCE:", error)
print("Derivada:", gradient)

np.testing.assert_allclose(error, -2 * np.log(0.75))
np.testing.assert_allclose(gradient, [4 / 3, -4 / 3])

print("Teste da BinaryCrossEntropy passou.")

Erro BCE: 0.5753641449035618
Derivada: [ 1.33333333 -1.33333333]
Teste da BinaryCrossEntropy passou.


In [9]:
import numpy as np

from si.neural_networks.optimizers import SGD

w = np.array([1.0, -1.0])
gradient = np.array([1.0, -2.0])

# Sem momentum
sgd = SGD(learning_rate=0.1, momentum=0.0)
updated = sgd.update(w, gradient)

np.testing.assert_allclose(updated, [0.9, -0.8])
print("Sem momentum:", updated)

# Com momentum: duas atualizações consecutivas
sgd_momentum = SGD(learning_rate=0.1, momentum=0.9)

first = sgd_momentum.update(w, gradient)
second = sgd_momentum.update(first, gradient)

np.testing.assert_allclose(first, [0.99, -0.98])
np.testing.assert_allclose(second, [0.971, -0.942])

print("Primeira atualização com momentum:", first)
print("Segunda atualização com momentum:", second)
print("Testes do SGD passaram.")

Sem momentum: [ 0.9 -0.8]
Primeira atualização com momentum: [ 0.99 -0.98]
Segunda atualização com momentum: [ 0.971 -0.942]
Testes do SGD passaram.


In [10]:
from si.neural_networks.neural_network import NeuralNetwork

print("NeuralNetwork importada com sucesso.")

NeuralNetwork importada com sucesso.


In [11]:
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import SigmoidActivation

net = NeuralNetwork()

net.add(DenseLayer(3, (2,)))
net.add(SigmoidActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())

for layer in net.layers:
    print(
        layer.layer_name(),
        layer.input_shape(),
        "->",
        layer.output_shape()
    )

assert len(net.layers) == 4
assert net.layers[0].weights.shape == (2, 3)
assert net.layers[2].weights.shape == (3, 1)

print("Teste do add passou.")

DenseLayer (2,) -> (3,)
SigmoidActivation (3,) -> (3,)
DenseLayer (3,) -> (1,)
SigmoidActivation (1,) -> (1,)
Teste do add passou.


In [12]:
import numpy as np

from si.neural_networks.neural_network import NeuralNetwork

# Dez amostras artificiais, com duas características cada
X = np.arange(20).reshape(10, 2)
y = np.arange(10)

net_batches = NeuralNetwork(batch_size=4)

lotes = list(
    net_batches._get_mini_batches(X, y, shuffle=False)
)

tamanhos = [len(X_lote) for X_lote, _ in lotes]

print("Tamanhos dos lotes:", tamanhos)
print("Total de amostras:", sum(tamanhos))

assert tamanhos == [4, 4, 2]

np.testing.assert_array_equal(
    np.concatenate([lote[0] for lote in lotes]), X
)
np.testing.assert_array_equal(
    np.concatenate([lote[1] for lote in lotes]), y
)

print("Teste dos mini-batches passou.")

Tamanhos dos lotes: [4, 4, 2]
Total de amostras: 10
Teste dos mini-batches passou.


In [13]:
import numpy as np

from si.data.dataset import Dataset
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer

dataset_score = Dataset(
    X=np.array([[1.0], [2.0]]),
    y=np.array([1.0, 2.0])
)

net_score = NeuralNetwork()
net_score.add(DenseLayer(1, (1,)))

# Fixamos os parâmetros para conhecer o resultado esperado
net_score.layers[0].weights[:] = 1.0
net_score.layers[0].biases[:] = 0.0

error1 = net_score.score(dataset_score)
print("MSE com peso 1:", error1)
np.testing.assert_allclose(error1, 0.0)

net_score.layers[0].weights[:] = 2.0

error2 = net_score.score(dataset_score)
print("MSE com peso 2:", error2)
np.testing.assert_allclose(error2, 2.5)

print("Teste do score com MSE passou.")

MSE com peso 1: 0.0
MSE com peso 2: 2.5
Teste do score com MSE passou.


In [14]:
import numpy as np

from si.metrics.accuracy import accuracy


def binary_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()

    classes = (
        np.asarray(y_pred).ravel() >= 0.5
    ).astype(int)

    return accuracy(y_true, classes)


# Quatro classificações corretas
y_true = np.array([0, 1, 1, 0])
probabilities = np.array([[0.1], [0.9], [0.8], [0.2]])

result1 = binary_accuracy(y_true, probabilities)

print("Accuracy com 4 corretas:", result1)
np.testing.assert_allclose(result1, 1.0)

# Alteramos a primeira previsão para ficar errada
probabilities[0] = 0.9

result2 = binary_accuracy(y_true, probabilities)

print("Accuracy com 3 corretas:", result2)
np.testing.assert_allclose(result2, 0.75)

print("Teste da accuracy binária passou.")

Accuracy com 4 corretas: 1.0
Accuracy com 3 corretas: 0.75
Teste da accuracy binária passou.


In [15]:
import numpy as np

from si.data.dataset import Dataset
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import MeanSquaredError

X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_xor = np.array([0, 1, 1, 0], dtype=float)

ds_xor = Dataset(X_xor, y_xor)

np.random.seed(1)

net_xor = NeuralNetwork(
    epochs=5000,
    batch_size=4,
    optimizer=SGD,
    learning_rate=0.5,
    loss=MeanSquaredError,
    metric=binary_accuracy,
    verbose=False
)

net_xor.add(DenseLayer(4, (2,)))
net_xor.add(SigmoidActivation())
net_xor.add(DenseLayer(1))
net_xor.add(SigmoidActivation())

net_xor.fit(ds_xor)

probabilities = net_xor.predict(ds_xor).ravel()
classes = (probabilities >= 0.5).astype(int)

print("Probabilidades:", probabilities.round(2))
print("Classes:", classes)
print("Accuracy:", net_xor.score(ds_xor))
print("MSE inicial:", net_xor.history[1]["loss"])
print("MSE final:", net_xor.history[5000]["loss"])

Probabilidades: [0.05 0.95 0.95 0.07]
Classes: [0 1 1 0]
Accuracy: 1.0
MSE inicial: 0.2500011450113375
MSE final: 0.0029973241506074644


In [16]:
np.random.seed(1)

net_sem_intermedia = NeuralNetwork(
    epochs=5000,
    batch_size=4,
    optimizer=SGD,
    learning_rate=0.5,
    loss=MeanSquaredError,
    metric=binary_accuracy,
    verbose=False
)

net_sem_intermedia.add(DenseLayer(4, (2,)))
net_sem_intermedia.add(DenseLayer(1))
net_sem_intermedia.add(SigmoidActivation())

net_sem_intermedia.fit(ds_xor)

prob_sem_intermedia = net_sem_intermedia.predict(ds_xor).ravel()

print("Probabilidades:", prob_sem_intermedia.round(2))
print("Accuracy:", net_sem_intermedia.score(ds_xor))
print("MSE final:", net_sem_intermedia.history[5000]["loss"])

Probabilidades: [0.5 0.5 0.5 0.5]
Accuracy: 0.5
MSE final: 0.24999999999999997


In [17]:
np.random.seed(1)

net_lr_baixa = NeuralNetwork(
    epochs=5000,
    batch_size=4,
    optimizer=SGD,
    learning_rate=0.01,
    loss=MeanSquaredError,
    metric=binary_accuracy,
    verbose=False
)

net_lr_baixa.add(DenseLayer(4, (2,)))
net_lr_baixa.add(SigmoidActivation())
net_lr_baixa.add(DenseLayer(1))
net_lr_baixa.add(SigmoidActivation())

net_lr_baixa.fit(ds_xor)

prob_lr_baixa = net_lr_baixa.predict(ds_xor).ravel()

print("Probabilidades:", prob_lr_baixa.round(4))
print("Accuracy:", net_lr_baixa.score(ds_xor))
print("MSE final:", net_lr_baixa.history[5000]["loss"])

Probabilidades: [0.4979 0.4997 0.5001 0.5017]
Accuracy: 0.5
MSE final: 0.24994853567513367


In [18]:
np.random.seed(0)

net_seed0 = NeuralNetwork(
    epochs=5000,
    batch_size=4,
    optimizer=SGD,
    learning_rate=0.5,
    loss=MeanSquaredError,
    metric=binary_accuracy,
    verbose=False
)

net_seed0.add(DenseLayer(4, (2,)))
net_seed0.add(SigmoidActivation())
net_seed0.add(DenseLayer(1))
net_seed0.add(SigmoidActivation())

net_seed0.fit(ds_xor)

prob_seed0 = net_seed0.predict(ds_xor).ravel()

print("Probabilidades:", prob_seed0.round(4))
print("Accuracy:", net_seed0.score(ds_xor))
print("MSE final:", net_seed0.history[5000]["loss"])

Probabilidades: [0.5002 0.4997 0.5003 0.4999]
Accuracy: 0.5
MSE final: 0.24999897109609653


In [19]:
import numpy as np

from si.data.dataset import Dataset

np.random.seed(42)

# 500 amostras, cada uma com 32 características
X_ex16 = np.random.randn(500, 32)

# Relação que determina a classe de cada amostra
combinacao = (
    X_ex16[:, 0]
    + 0.8 * X_ex16[:, 1]
    - 0.6 * X_ex16[:, 2]
    + 0.5 * X_ex16[:, 3]
)

# Combinação positiva: classe 1; restantes casos: classe 0
y_ex16 = (combinacao > 0).astype(int)

ds_ex16 = Dataset(X_ex16, y_ex16)

print("Dimensão de X:", ds_ex16.X.shape)
print("Dimensão de y:", ds_ex16.y.shape)
print("Classes:", np.unique(ds_ex16.y))

Dimensão de X: (500, 32)
Dimensão de y: (500,)
Classes: [0 1]


In [20]:
import numpy as np

from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy


def criar_rede_ex16(seed=1):
    np.random.seed(seed)

    rede = NeuralNetwork(
        epochs=100,
        batch_size=16,
        optimizer=SGD,
        learning_rate=0.01,
        loss=BinaryCrossEntropy,
        metric=binary_accuracy,
        verbose=False
    )

    rede.add(DenseLayer(16, (32,)))
    rede.add(ReLUActivation())

    rede.add(DenseLayer(8))
    rede.add(ReLUActivation())

    rede.add(DenseLayer(1))
    rede.add(SigmoidActivation())

    return rede


rede_ex16 = criar_rede_ex16()

for camada in rede_ex16.layers:
    print(
        camada.layer_name(),
        camada.input_shape(),
        "->",
        camada.output_shape()
    )

DenseLayer (32,) -> (16,)
ReLUActivation (16,) -> (16,)
DenseLayer (16,) -> (8,)
ReLUActivation (8,) -> (8,)
DenseLayer (8,) -> (1,)
SigmoidActivation (1,) -> (1,)


In [21]:
import numpy as np

from si.data.dataset import Dataset

k = 5

np.random.seed(42)
indices = np.random.permutation(ds_ex16.X.shape[0])
partes = np.array_split(indices, k)

accuracies_ex16 = []

for i in range(k):
    indices_teste = partes[i]

    partes_treino = []

    for j in range(k):
        if j != i:
            partes_treino.append(partes[j])

    indices_treino = np.concatenate(partes_treino)

    ds_treino = Dataset(
        ds_ex16.X[indices_treino],
        ds_ex16.y[indices_treino]
    )

    ds_teste = Dataset(
        ds_ex16.X[indices_teste],
        ds_ex16.y[indices_teste]
    )

    # Uma rede nova para cada fold
    rede_fold = criar_rede_ex16(seed=i + 1)

    rede_fold.fit(ds_treino)
    acc = rede_fold.score(ds_teste)

    accuracies_ex16.append(acc)

    print(f"Fold {i + 1}: accuracy = {acc:.4f}")

media_ex16 = np.mean(accuracies_ex16)
desvio_ex16 = np.std(accuracies_ex16)

print(f"\nAccuracy média: {media_ex16:.4f}")
print(f"Desvio-padrão: {desvio_ex16:.4f}")

Fold 1: accuracy = 0.9800
Fold 2: accuracy = 0.8800
Fold 3: accuracy = 0.9300
Fold 4: accuracy = 0.9000
Fold 5: accuracy = 0.9000

Accuracy média: 0.9180
Desvio-padrão: 0.0349


In [22]:
import numpy as np

from si.data.dataset import Dataset
from si.models.knn_classifier import KNNClassifier
from si.model_selection.grid_search import grid_search_cv

# Dois grupos claramente separados
X = np.array([
    [0.0], [0.1], [0.2], [0.3],
    [10.0], [10.1], [10.2], [10.3]
])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

ds_verificacao = Dataset(X, y)

resultados = grid_search_cv(
    model=KNNClassifier(),
    dataset=ds_verificacao,
    hyperparameter_grid={"k": [1, 6]},
    cv=4,
    seed=1
)

print("Scores:", resultados["scores"])
print("Melhor configuração:", resultados["best_hyperparameters"])
print("Melhor score:", resultados["best_score"])

np.testing.assert_allclose(resultados["scores"], [1.0, 0.5])
assert resultados["best_hyperparameters"] == {"k": 1}

print("Verificação passou.")

Scores: [1.0, 0.5]
Melhor configuração: {'k': 1}
Melhor score: 1.0
Verificação passou.


In [23]:
import numpy as np

from si.metrics.accuracy import accuracy
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy


def binary_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    classes = (np.asarray(y_pred).ravel() >= 0.5).astype(int)

    return accuracy(y_true, classes)


def build_network(learning_rate, n_hidden):
    np.random.seed(0)

    net = NeuralNetwork(
        epochs=100,
        batch_size=16,
        optimizer=SGD,
        learning_rate=learning_rate,
        momentum=0.9,
        loss=BinaryCrossEntropy,
        metric=binary_accuracy,
        verbose=False
    )

    net.add(DenseLayer(n_hidden, (2,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())

    return net

In [24]:
rng = np.random.default_rng(42)

X_circulo = rng.uniform(-2, 2, size=(400, 2))

y_circulo = (
    X_circulo[:, 0] ** 2 + X_circulo[:, 1] ** 2 < 1.5
).astype(float)

ds_circulo = Dataset(X_circulo, y_circulo)

grid_circulo = {
    "learning_rate": [0.001, 0.01, 0.1],
    "n_hidden": [2, 4, 16]
}

resultados_circulo = grid_search_cv(
    model=build_network,
    dataset=ds_circulo,
    hyperparameter_grid=grid_circulo,
    cv=5,
    seed=1
)

for configuracao, score in zip(
    resultados_circulo["hyperparameters"],
    resultados_circulo["scores"]
):
    print(configuracao, round(score, 3))

print(
    "\nMelhor configuração:",
    resultados_circulo["best_hyperparameters"]
)
print(
    "Melhor accuracy média:",
    resultados_circulo["best_score"]
)

{'learning_rate': 0.001, 'n_hidden': 2} 0.715
{'learning_rate': 0.001, 'n_hidden': 4} 0.715
{'learning_rate': 0.001, 'n_hidden': 16} 0.843
{'learning_rate': 0.01, 'n_hidden': 2} 0.727
{'learning_rate': 0.01, 'n_hidden': 4} 0.722
{'learning_rate': 0.01, 'n_hidden': 16} 0.975
{'learning_rate': 0.1, 'n_hidden': 2} 0.713
{'learning_rate': 0.1, 'n_hidden': 4} 0.765
{'learning_rate': 0.1, 'n_hidden': 16} 0.98

Melhor configuração: {'learning_rate': 0.1, 'n_hidden': 16}
Melhor accuracy média: 0.9800000000000001


In [25]:
from si.model_selection.randomized_search import log_uniform

rng = np.random.default_rng(42)
sortear = log_uniform(1e-4, 1e-1)

valores = [sortear(rng) for _ in range(5)]

print(valores)

assert all(1e-4 <= valor < 1e-1 for valor in valores)

print("Verificação de log_uniform passou.")

[0.02098302729986613, 0.0020731719263694525, 0.03765249501831186, 0.012362436879527079, 0.00019166024706981723]
Verificação de log_uniform passou.


In [26]:
import importlib
import si.model_selection.randomized_search as randomized_search_module

# Carregar a versão que acabaste de guardar
importlib.reload(randomized_search_module)

from si.model_selection.randomized_search import randomized_search_cv

resultados_random = randomized_search_cv(
    model=KNNClassifier(),
    dataset=ds_verificacao,
    hyperparameter_distributions={"k": [1, 6]},
    cv=4,
    n_iter=6,
    seed=1
)

assert (
    len(resultados_random["hyperparameters"])
    == len(resultados_random["scores"])
    == 6
)

for configuracao, score in zip(
    resultados_random["hyperparameters"],
    resultados_random["scores"]
):
    print(configuracao, score)

    esperado = 1.0 if configuracao["k"] == 1 else 0.5
    np.testing.assert_allclose(score, esperado)

assert resultados_random["best_hyperparameters"] == {"k": 1}
np.testing.assert_allclose(resultados_random["best_score"], 1.0)

print("Melhor configuração:", resultados_random["best_hyperparameters"])
print("Melhor score:", resultados_random["best_score"])
print("Verificação de randomized_search_cv passou.")

{'k': 6} 0.5
{'k': 1} 1.0
{'k': 1} 1.0
{'k': 6} 0.5
{'k': 1} 1.0
{'k': 1} 1.0
Melhor configuração: {'k': 1}
Melhor score: 1.0
Verificação de randomized_search_cv passou.


In [27]:
from si.model_selection.split import train_test_split

# Recriar os mesmos dados do exercício 16
np.random.seed(42)

X_ex16 = np.random.randn(500, 32)

combinacao = (
    X_ex16[:, 0]
    + 0.8 * X_ex16[:, 1]
    - 0.6 * X_ex16[:, 2]
    + 0.5 * X_ex16[:, 3]
)

y_ex16 = (combinacao > 0).astype(int)

ds_ex16 = Dataset(X_ex16, y_ex16)

# Separar treino e teste
ds_treino_ex18, ds_teste_ex18 = train_test_split(
    ds_ex16,
    test_size=0.2,
    random_state=42
)

print("Treino:", ds_treino_ex18.X.shape)
print("Teste:", ds_teste_ex18.X.shape)

Treino: (400, 32)
Teste: (100, 32)


In [28]:
def build_network_ex18(learning_rate, batch_size, epochs):
    np.random.seed(1)

    net = NeuralNetwork(
        epochs=epochs,
        batch_size=batch_size,
        optimizer=SGD,
        learning_rate=learning_rate,
        loss=BinaryCrossEntropy,
        metric=binary_accuracy,
        verbose=False
    )

    net.add(DenseLayer(16, (32,)))
    net.add(ReLUActivation())

    net.add(DenseLayer(8))
    net.add(ReLUActivation())

    net.add(DenseLayer(1))
    net.add(SigmoidActivation())

    return net

In [29]:
from si.model_selection.randomized_search import (
    log_uniform,
    randomized_search_cv
)

distribuicoes_ex18 = {
    "learning_rate": log_uniform(1e-4, 1e-1),
    "batch_size": [8, 16, 32, 64],
    "epochs": [50, 100, 200]
}

resultados_ex18 = randomized_search_cv(
    model=build_network_ex18,
    dataset=ds_treino_ex18,
    hyperparameter_distributions=distribuicoes_ex18,
    scoring=binary_accuracy,
    cv=3,
    n_iter=20,
    seed=42
)

print(
    "Melhor configuração:",
    resultados_ex18["best_hyperparameters"]
)

print(
    "Melhor accuracy média de validação:",
    resultados_ex18["best_score"]
)

Melhor configuração: {'learning_rate': 0.06424206139428561, 'batch_size': 8, 'epochs': 200}
Melhor accuracy média de validação: 0.895092208132271


In [34]:
import numpy as np

configuracoes = {
    "Modelo otimizado": {
        "learning_rate": 0.06424206139428561,
        "batch_size": 8,
        "epochs": 200
    },
    "Modelo do exercício 16": {
        "learning_rate": 0.01,
        "batch_size": 16,
        "epochs": 100
    }
}

for nome, parametros in configuracoes.items():
    # Criar uma rede nova e treinar
    modelo = build_network_ex18(**parametros)
    modelo.fit(ds_treino_ex18)

    # Prever as classes no conjunto de teste
    probabilidades = modelo.predict(ds_teste_ex18).ravel()
    previsoes = (probabilidades >= 0.5).astype(int)

    # Calcular a accuracy de teste
    acc = np.mean(previsoes == ds_teste_ex18.y.ravel())

    print(f"{nome} — accuracy de teste: {acc:.2%}")

Modelo otimizado — accuracy de teste: 96.00%
Modelo do exercício 16 — accuracy de teste: 98.00%
